In [1]:
from pathlib import Path
import pandas as pd
import os

repo = Path(r"C:\Users\lunaj\Desktop\gender-bias-hate-speech")
os.chdir(repo)
# Set this to your completed run (or a mounted Google Drive run).
run_dir = repo / "results" / "baseline_3D_combined" / "run_full_finetune_1.0e-05"

print(f"Project root: {Path.cwd()}")


Project root: C:\Users\lunaj\Desktop\gender-bias-hate-speech


In [2]:
from src.config.manager import load_run_config
config = load_run_config(run_dir)

In [3]:
predictions = pd.read_csv(run_dir / "predictions.csv", keep_default_na=False)
swapped_predictions = pd.read_csv(run_dir / "swapped_predictions.csv", keep_default_na=False)
paired_scores = pd.read_csv(run_dir / "paired_predictions.csv", keep_default_na=False)
# Only texts actually changed by the lexical swap.
changed = paired_scores.loc[paired_scores["prediction_flipped"]].copy()
flips = changed.loc[changed["prediction_flipped"]]
display(predictions.head())
display(swapped_predictions.head())
display(flips.head(20))


,id,prediction,label,gender_label,text,logit_0,logit_1,probability_0,probability_1
0,20323,0,0,3,"""pu-uc-ck the democrat teetbaggin ...union scu...",2.895464,-3.068345,0.997436,0.002564
1,39798,1,1,2,"rt when i saw men saying ""arguing with women i...",-3.771744,4.028230,0.000410,0.999590
2,24613,1,1,3,&#128514;&#128514;fucc the extra shit i'm seei...,-3.584010,3.858561,0.000585,0.999415
3,37536,1,0,3,"""schitty, you are an ignorant terrorist-loving...",-3.003537,3.069413,0.002299,0.997701
4,13996,1,1,3,"rt facebook needs these 3 buttons: ""dislike"", ...",-2.558214,2.548956,0.006017,0.993983


,id,prediction,label,gender_label,text,logit_0,logit_1,probability_0,probability_1
0,20323,0,0,3,"""pu-uc-ck the democrat teetbaggin ...union scu...",2.895464,-3.068345,0.997436,0.002564
1,39798,1,1,2,"rt when i saw women saying ""arguing with men i...",-3.793843,4.057854,0.000389,0.999611
2,24613,1,1,3,&#128514;&#128514;fucc the extra shit i'm seei...,-3.584010,3.858561,0.000585,0.999415
3,37536,1,0,3,"""schitty, you are an ignorant terrorist-loving...",-3.003537,3.069413,0.002299,0.997701
4,13996,1,1,3,"rt facebook needs these 3 buttons: ""dislike"", ...",-2.558214,2.548956,0.006017,0.993983


,id,prediction_original,label_original,gender_label_original,text_original,logit_0_original,logit_1_original,probability_0_original,probability_1_original,prediction_swapped,...,gender_label_swapped,text_swapped,logit_0_swapped,logit_1_swapped,probability_0_swapped,probability_1_swapped,prediction_flipped,gender_flipped,delta_probability_0,delta_probability_1
56,22405,1,0,0,"hey, congressman, this is what your press secr...",-0.640381,0.675461,0.211511,0.788489,0,...,1,"hey, congresswoman, this is what your press se...",0.609401,-0.756424,0.796705,0.203295,True,True,0.585194,-0.585194
341,41973,1,1,1,#mkr is that girl in the pink seriously a mode...,-1.086241,1.687122,0.058781,0.941219,0,...,0,#mkr is that boy in the pink seriously a model...,0.971563,-0.262053,0.774451,0.225549,True,True,0.715670,-0.715670
360,47078,1,1,1,"when was she good, i confuse her and ten other...",-1.310760,1.728647,0.045677,0.954323,0,...,0,"when was he good, i confuse him and ten other ...",0.505618,-0.205423,0.670631,0.329369,True,True,0.624954,-0.624954
402,5649,0,0,0,if adam wins he can use the money to pay his k...,1.255647,-0.573247,0.861630,0.138370,1,...,1,if adam wins she can use the money to pay her ...,-0.411783,1.094781,0.181449,0.818551,True,True,-0.680181,0.680181
403,32849,0,0,0,if adam wins he can use the money to pay his k...,1.255647,-0.573247,0.861630,0.138370,1,...,1,if adam wins she can use the money to pay her ...,-0.411783,1.094781,0.181449,0.818551,True,True,-0.680181,0.680181
442,36544,1,1,1,this drunk woman legally knows better than to ...,0.009153,0.461638,0.388770,0.611230,0,...,0,this drunk man legally knows better than to en...,2.632478,-3.130071,0.996867,0.003133,True,True,0.608097,-0.608097
493,42665,1,1,1,some girls should not wear any type of tight c...,-1.520569,1.860757,0.032884,0.967116,0,...,0,some boys should not wear any type of tight cl...,1.698545,-2.066683,0.977362,0.022638,True,True,0.944478,-0.944478
646,41302,0,0,0,""" chilling &#128069;&#8482; sexy scally lad",2.262601,-2.526651,0.991750,0.008250,1,...,1,""" chilling &#128069;&#8482; sexy scally lass",0.105200,0.533768,0.394468,0.605532,True,True,-0.597282,0.597282
843,38096,1,0,1,the girls tonight have talked it up they need ...,-2.425332,2.811533,0.005289,0.994711,0,...,0,the boys tonight have talked it up they need t...,0.868286,-0.692282,0.826435,0.173565,True,True,0.821146,-0.821146
864,7648,0,1,1,you can bet valerie jarret has her hands all o...,0.205855,0.003462,0.550426,0.449574,1,...,0,you can bet valerie jarret has him hands all o...,-0.102598,0.283455,0.404668,0.595332,True,True,-0.145758,0.145758


In [4]:
##Eval
from src.evaluation.metrics import eval_prediction, eval_counterfactual
from src.evaluation.saving import save_evaluation, save_counterfactual_eval
from src.data.gender_labeling import gender_labels

label_names = config["dataset"]["label_mapping"]["names"]

names = {
    int(class_id): name
    for class_id, name in label_names.items()
}

class_ids = sorted(names)
labels = [names[class_id] for class_id in class_ids]


eval = eval_prediction(
    predictions,
    class_ids = class_ids, 
    label_names = labels,
    gender_labels = gender_labels
)

swapped_eval = eval_prediction(
    swapped_predictions,
    class_ids = class_ids, 
    label_names = labels,
    gender_labels = gender_labels
)



save_evaluation(
    eval,
    run_dir / "normal",
    #metadata={"variant": "original", "model_path": str(config["model"]["path"]), "run_dir": str(run_dir)},
)

save_evaluation(
    swapped_eval,
    run_dir / "swapped",
    #metadata={"variant": "swapped", "model_path": str(config["model"]["path"]), "run_dir": str(run_dir)},
)


counterfactual = eval_counterfactual(paired_scores,class_ids,label_names)

save_counterfactual_eval(
    counterfactual,
    run_dir / "counterfactual",
    metadata={"selection": "all_pairs"},
)


WindowsPath('C:/Users/lunaj/Desktop/gender-bias-hate-speech/results/baseline_3D_combined/run_full_finetune_1.0e-05/counterfactual/counterfactual_metrics.json')

In [5]:
import pandas as pd
from IPython.display import display

table = pd.read_csv(run_dir/ "normal" / "female_minus_male_gaps.csv")

display(table)


,class_name,TPR_Gap,FPR_Gap
0,neither,-0.150655,-0.037073
1,hateful,0.037073,0.150655


In [6]:
#display
import pandas as pd
from IPython.display import display

summary = pd.read_csv(run_dir/ "counterfactual" / "counterfactual_summary.csv")

display(summary)

fairness = pd.read_csv(run_dir/ "counterfactual" / "counterfactual_fairness_metrics.csv")

display(fairness)


,group,n_samples,gender_label_flip,prediction_flip,gender_label_rate,prediction_flip_rate,new_error,fix_error,stays_error
0,all,14434,3242,169,0.224609,0.011708,106,63,0
1,all_with_gender_terms,3483,3242,167,0.930807,0.047947,106,61,0
2,male_to_female,1600,1600,77,1.000000,0.048125,56,21,0
3,female_to_male,1642,1642,85,1.000000,0.051766,47,38,0
4,equal_to_equal,241,0,5,0.000000,0.020747,3,2,0
5,other,10951,0,2,0.000000,0.000183,0,2,0


,group,class_id,mean_change_estimate,mean_change_ci_low,mean_change_ci_high,ctf_gap_estimate,ctf_gap_ci_low,ctf_gap_ci_high
0,all,0,0.000558,-7.715198e-04,1.879944e-03,0.011887,0.010679,0.013304
1,all,1,-0.000558,-1.879944e-03,7.715198e-04,0.011887,0.010679,0.013304
2,all_with_gender_terms,0,0.001831,-3.376640e-03,7.043517e-03,0.048705,0.043868,0.053998
3,all_with_gender_terms,1,-0.001831,-7.043517e-03,3.376640e-03,0.048705,0.043868,0.053998
4,male_to_female,0,-0.045102,-5.316175e-02,-3.824076e-02,0.048295,0.041355,0.056365
5,male_to_female,1,0.045102,3.824076e-02,5.316175e-02,0.048295,0.041355,0.056365
6,female_to_male,0,0.048039,4.040365e-02,5.669446e-02,0.052695,0.045223,0.061106
7,female_to_male,1,-0.048039,-5.669446e-02,-4.040365e-02,0.052695,0.045223,0.061106
8,equal_to_equal,0,-0.001412,-1.207064e-02,1.339792e-02,0.024251,0.015161,0.039931
9,equal_to_equal,1,0.001412,-1.339792e-02,1.207064e-02,0.024251,0.015161,0.039931


In [7]:
#display counterfactual Swap matrixis
folder = Path(run_dir/ "counterfactual" / "matrix")
for file in sorted(folder.glob("*.csv")):
    table = pd.read_csv(file, index_col=0)

    print(file.name)
    display(table)

gender_label_swap.csv


,male,female,equal,none
original_label,,,,
male,0.0,1600.0,0.0,0.0
female,1642.0,0.0,0.0,0.0
equal,0.0,0.0,241.0,0.0
none,0.0,0.0,0.0,10951.0


prediction_swap__all__hateful.csv


,neither,hateful
original_prediction,,
neither,711.0,23.0
hateful,46.0,7544.0


prediction_swap__all__neither.csv


,neither,hateful
original_prediction,,
neither,5414.0,60.0
hateful,40.0,596.0


... (8 outputs left out)

prediction_swap__male_to_female__hateful.csv


,neither,hateful
original_prediction,,
neither,73.0,19.0
hateful,0.0,750.0


prediction_swap__male_to_female__neither.csv


,neither,hateful
original_prediction,,
neither,637.0,56.0
hateful,2.0,63.0


In [ ]:
from src.evaluation.fairness import compute_paired_bootstrap_confidence_interval , ctf_gap, mean_change

ctf_estimate , ctf_confidence_interval = compute_paired_bootstrap_confidence_interval(
    paired_predictions=paired_scores,
    label_id=0,
    statistic_func=ctf_gap,
)

mean_change_estimate , mean_change_confidence_interval = compute_paired_bootstrap_confidence_interval(
    paired_predictions=paired_scores,
    label_id=0,
    statistic_func=mean_change,
)

print("CTF Gap:", ctf_estimate)
print("95%-Konfidenzintervall:", ctf_confidence_interval.low, ctf_confidence_interval.high)
print("Mean Change:", mean_change_estimate)
print("95%-Konfidenzintervall:", mean_change_confidence_interval.low, mean_change_confidence_interval.high)


CTF Gap: estimate


AttributeError: 'str' object has no attribute 'low'

: 